Lucas Cordeiro Raw - 24007079

In [ ]:
!pip install statstests

In [ ]:
import pandas as pd #manipulação de dados em formato de dataframe
import seaborn as sns #biblioteca de visualização de informações estatísticas
import matplotlib.pyplot as plt #biblioteca de visualização de dados
import statsmodels.api as sm #biblioteca de modelagem estatística
import numpy as np #biblioteca para operações matemáticas multidimensionais
from scipy import stats #testes estatísticos
from statsmodels.iolib.summary2 import summary_col #comparação entre modelos
import plotly.graph_objs as go #gráfico 3D
from scipy.stats import pearsonr #correlações de Pearson
from statstests.process import stepwise #procedimento Stepwise
from scipy.stats import shapiro #teste de Shapiro-Wilk
from statstests.tests import shapiro_francia #teste de Shapiro-Francia
from statsmodels.stats.outliers_influence import variance_inflation_factor #diagnóstico de multicolinearidade
from statsmodels.stats.stattools import durbin_watson #teste de Durbin-Watson
import statsmodels.stats.diagnostic as dg #teste de Breusch-Godfrey
from scipy.stats import norm #para plotagem da distribuição normal no histograma
import statsmodels.formula.api as smf #regressão quantílica

# Questao 1

In [ ]:
df= pd.read_csv('corrupcao.csv')
print(df)

                    pais  cpi  idade      horas
0              Argentina  2.9     72  35.000000
1              Australia  8.7     64  32.000000
2                Austria  7.9     72  32.000000
3                Belgium  7.1     67  30.100000
4                 Brazil  3.7     59  35.000000
5                 Canada  8.9     61  33.400002
6                  Chile  7.2     70  34.000000
7                  China  3.5     49  34.000000
8               Colombia  3.5     79  33.000000
9                 Cyprus  6.3     58  32.000000
10            Czech Rep.  4.6     42  38.099998
11               Denmark  9.3     76  30.000000
12                 Egypt  3.1     59  32.000000
13                France  6.8     70  30.000000
14               Germany  7.9     66  27.500000
15                Greece  3.5     60  30.000000
16               Iceland  8.5     53  31.000000
17                 India  3.3     56  29.799999
18             Indonesia  2.8     63  33.500000
19               Ireland  8.0     63  31

In [ ]:
x = df[['idade', 'horas']]
x = sm.add_constant(x)
y = df['cpi']
modelo = sm.OLS(y, x).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                    cpi   R-squared:                       0.318
Model:                            OLS   Adj. R-squared:                  0.290
Method:                 Least Squares   F-statistic:                     11.41
Date:                Fri, 27 Mar 2026   Prob (F-statistic):           8.55e-05
Time:                        11:59:12   Log-Likelihood:                -107.81
No. Observations:                  52   AIC:                             221.6
Df Residuals:                      49   BIC:                             227.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         15.1589      4.754      3.188      0.0

In [ ]:
#1a - Teste F
if modelo.f_pvalue < 0.05:
  print('é estatisticamente significativo')
  print(modelo.f_pvalue)
else:
  print('não é estatisticamente significativo')

é estatisticamente significativo
8.54915962282292e-05


In [ ]:
#1b - Teste t (variaveis individuais)
if modelo.pvalues[1] < 0.05:
  print('é estatisticamente significativo')
  print(modelo.pvalues)
else:
  print('não é estatisticamente significativo')

é estatisticamente significativo
const    0.002493
idade    0.037568
horas    0.000674
dtype: float64


/tmp/ipykernel_1591/2881156015.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if modelo.pvalues[1] < 0.05:


In [ ]:
#1c - Equação estimada
cpi = 15.1589 + 0.0701 * df['idade'] - 0.0002 * df['horas']

In [ ]:
#1d - Qual o R²?
print(modelo.rsquared)


0.3177303412103135


A variável idade apresenta coeficiente positivo, indicando que, mantidas as demais variáveis constantes, um aumento na idade média dos bilionários está associado a um aumento no índice CPI.

Por outro lado, a variável horas apresenta coeficiente negativo, sugerindo que, mantidas as demais variáveis constantes, um aumento na quantidade média de horas trabalhadas semanalmente está associado a uma redução no índice CPI.

Dessa forma, conclui-se que países com bilionários mais velhos tendem a apresentar maiores níveis de CPI, enquanto países com maior carga de trabalho semanal tendem a apresentar menores níveis desse índice.

# Questao 2

In [ ]:
df_emer = pd.read_csv('corrupcaoemer.csv')
df_emer

,pais,cpi,idade,horas,emergente
0,Argentina,2.9,72,35.000000,Emergente
1,Australia,8.7,64,32.000000,Desenvolvido
2,Austria,7.9,72,32.000000,Desenvolvido
3,Belgium,7.1,67,30.100000,Desenvolvido
4,Brazil,3.7,59,35.000000,Emergente
5,Canada,8.9,61,33.400002,Desenvolvido
6,Chile,7.2,70,34.000000,Emergente
7,China,3.5,49,34.000000,Emergente
8,Colombia,3.5,79,33.000000,Emergente
9,Cyprus,6.3,58,32.000000,Emergente


In [ ]:
#2a - média emergentes e
grupo_emerg = df_emer[df_emer['emergente'] == 'Emergente']['cpi']
grupo_desv =  df_emer[df_emer['emergente'] == 'Desenvolvido']['cpi']

print('Media emergentes', grupo_emerg.mean())
print('Media desenvolvidos', grupo_desv.mean())

t_stat, p_val = stats.ttest_ind(grupo_emerg, grupo_desv)
print('\np-valor', p_val , '- É significativo.')

Media emergentes 4.096774225806452
Media desenvolvidos 7.728571438095238

p-valor 3.976807885703568e-11 - É significativo.


In [ ]:
#2b - Modelo final estimado
modelo2 = smf.ols('cpi ~ idade + horas + C(emergente)', data=df_emer).fit()
print(modelo2.summary())

                            OLS Regression Results                            
Dep. Variable:                    cpi   R-squared:                       0.623
Model:                            OLS   Adj. R-squared:                  0.599
Method:                 Least Squares   F-statistic:                     26.45
Date:                Fri, 27 Mar 2026   Prob (F-statistic):           3.04e-10
Time:                        12:42:12   Log-Likelihood:                -92.379
No. Observations:                  52   AIC:                             192.8
Df Residuals:                      48   BIC:                             200.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [ ]:
#2c - Previsão
novo_df = pd.DataFrame({
    'const': [1]
    , 'idade': [51]
    , 'horas': [37]
    , 'emergente': ['Emergente']
})

prev = modelo2.predict(novo_df)
print(prev)

0    3.303782
dtype: float64


In [ ]:
#2d - Intervalo de confiança
prev_ = modelo2.get_prediction(novo_df)
intervalo = prev_.conf_int(alpha =0.5)

print("Intervalo: ", intervalo )

Intervalo:  [[2.99678926 3.61077511]]
